# Lab 1: NLP Preprocessing

In [24]:
def tokenize(sentence):
    return sentence.lower().split()

print(tokenize('I am a student'))

['i', 'am', 'a', 'student']


## Build Vocabulary with Special Tokens

Theo handout, chúng ta cần thêm **special tokens**:
- `<PAD>` (ID=0): Padding token - dùng để đệm các câu ngắn
- `<BOS>` (ID=1): Begin of Sentence - đánh dấu bắt đầu câu
- `<EOS>` (ID=2): End of Sentence - đánh dấu kết thúc câu
- Các từ thường bắt đầu từ ID=3

In [25]:
class Vocab:
    """Class để xây dựng và quản lý từ điển (vocabulary) với special tokens"""
    
    # Định nghĩa special tokens theo handout
    PAD_TOKEN = '<PAD>'
    BOS_TOKEN = '<BOS>'
    EOS_TOKEN = '<EOS>'
    
    def __init__(self):
        self.word2id = {}  # Dictionary ánh xạ từ -> ID
        self.id2word = {}  # Dictionary ánh xạ ID -> từ
        
        # Thêm special tokens ngay từ đầu với ID cố định
        # Theo handout: <PAD>=0, <BOS>=1, <EOS>=2
        self._add_special_tokens()
    
    def _add_special_tokens(self):
        """Thêm các special tokens với ID cố định"""
        self.word2id[self.PAD_TOKEN] = 0
        self.id2word[0] = self.PAD_TOKEN
        
        self.word2id[self.BOS_TOKEN] = 1
        self.id2word[1] = self.BOS_TOKEN
        
        self.word2id[self.EOS_TOKEN] = 2
        self.id2word[2] = self.EOS_TOKEN

    def add_word(self, w):
        """Thêm một từ vào từ điển nếu chưa tồn tại"""
        if w not in self.word2id:
            idx = len(self.word2id)  # ID mới = số từ hiện có
            self.word2id[w] = idx
            self.id2word[idx] = w
    
    def build_vocab(self, sentences):
        """Xây dựng từ điển từ danh sách các câu (special tokens đã có sẵn)"""
        for sentence in sentences:
            words = tokenize(sentence)  # Tách từ
            for word in words:
                self.add_word(word)  # Thêm từng từ vào vocab
    
    def encode(self, sentence, add_special_tokens=True):
        """
        Chuyển câu (string) thành list các ID (số)
        
        Args:
            sentence: Câu cần encode
            add_special_tokens: Nếu True, tự động thêm <BOS> và <EOS>
        
        Returns:
            List các ID, ví dụ: [1, 3, 4, 5, 2] = [<BOS>, i, am, student, <EOS>]
        """
        words = tokenize(sentence)
        ids = [self.word2id[word] for word in words if word in self.word2id]
        
        if add_special_tokens:
            # Thêm <BOS> ở đầu và <EOS> ở cuối
            ids = [self.word2id[self.BOS_TOKEN]] + ids + [self.word2id[self.EOS_TOKEN]]
        
        return ids
    
    def decode(self, ids, skip_special_tokens=True):
        """
        Chuyển list các ID (số) thành câu (string)
        
        Args:
            ids: List các ID
            skip_special_tokens: Nếu True, bỏ qua special tokens khi decode
        
        Returns:
            Câu văn bản
        """
        words = []
        for id in ids:
            if id in self.id2word:
                word = self.id2word[id]
                # Bỏ qua special tokens nếu cần
                if skip_special_tokens and word in [self.PAD_TOKEN, self.BOS_TOKEN, self.EOS_TOKEN]:
                    continue
                words.append(word)
        
        return ' '.join(words)
    
    def __len__(self):
        """Trả về kích thước của từ điển (bao gồm cả special tokens)"""
        return len(self.word2id)

## Test class Vocab với Special Tokens

Kiểm tra xem special tokens đã được thêm đúng chưa:

In [26]:
# Tạo từ điển test
vocab_test = Vocab()

print("=== Special Tokens (Đã tự động thêm) ===")
print(f"  '{vocab_test.PAD_TOKEN}' -> {vocab_test.word2id[vocab_test.PAD_TOKEN]}")
print(f"  '{vocab_test.BOS_TOKEN}' -> {vocab_test.word2id[vocab_test.BOS_TOKEN]}")
print(f"  '{vocab_test.EOS_TOKEN}' -> {vocab_test.word2id[vocab_test.EOS_TOKEN]}")

# Thêm một vài câu
test_sentences = [
    "I am a student",
    "I am a teacher",
    "She is a student"
]

# Xây dựng vocab
vocab_test.build_vocab(test_sentences)

print(f"\n=== Từ điển đã xây dựng ===")
print(f"Tổng số token (bao gồm special): {len(vocab_test)}")
print(f"Số từ vựng thực tế: {len(vocab_test) - 3}")

print(f"\n=== Toàn bộ word2id ===")
for word, idx in vocab_test.word2id.items():
    print(f"  '{word}' -> ID: {idx}")

=== Special Tokens (Đã tự động thêm) ===
  '<PAD>' -> 0
  '<BOS>' -> 1
  '<EOS>' -> 2

=== Từ điển đã xây dựng ===
Tổng số token (bao gồm special): 10
Số từ vựng thực tế: 7

=== Toàn bộ word2id ===
  '<PAD>' -> ID: 0
  '<BOS>' -> ID: 1
  '<EOS>' -> ID: 2
  'i' -> ID: 3
  'am' -> ID: 4
  'a' -> ID: 5
  'student' -> ID: 6
  'teacher' -> ID: 7
  'she' -> ID: 8
  'is' -> ID: 9


In [27]:
# Test encode và decode theo handout
test_sentence = "I am a student"
print(f"=== Test Encoding/Decoding (theo handout) ===")
print(f"Câu gốc: '{test_sentence}'")

# Encode WITH special tokens (mặc định theo handout)
encoded = vocab_test.encode(test_sentence, add_special_tokens=True)
print(f"\nEncoded (with special tokens): {encoded}")
print(f"Giải thích: [", end="")
for i, id in enumerate(encoded):
    word = vocab_test.id2word[id]
    if i > 0:
        print(", ", end="")
    print(f"{id}({word})", end="")
print("]")

# Decode - tự động bỏ qua special tokens
decoded = vocab_test.decode(encoded, skip_special_tokens=True)
print(f"\nDecoded (skip special): '{decoded}'")

# Decode - giữ nguyên special tokens
decoded_full = vocab_test.decode(encoded, skip_special_tokens=False)
print(f"Decoded (keep special):  '{decoded_full}'")

# Kiểm tra
if test_sentence.lower() == decoded:
    print("\n✓ Encode/Decode hoạt động đúng!")
else:
    print("\n✗ Có lỗi xảy ra")

print("\n" + "="*60)
print("CHÚ Ý: Theo handout, câu được encode thành:")
print("  [<BOS>, word1, word2, ..., <EOS>]")
print("Ví dụ: 'I am a student' -> [1, 3, 4, 5, 6, 2]")
print("="*60)

=== Test Encoding/Decoding (theo handout) ===
Câu gốc: 'I am a student'

Encoded (with special tokens): [1, 3, 4, 5, 6, 2]
Giải thích: [1(<BOS>), 3(i), 4(am), 5(a), 6(student), 2(<EOS>)]

Decoded (skip special): 'i am a student'
Decoded (keep special):  '<BOS> i am a student <EOS>'

✓ Encode/Decode hoạt động đúng!

CHÚ Ý: Theo handout, câu được encode thành:
  [<BOS>, word1, word2, ..., <EOS>]
Ví dụ: 'I am a student' -> [1, 3, 4, 5, 6, 2]


## Load dữ liệu thực (English-Vietnamese)

Bây giờ chúng ta sẽ load dữ liệu thực từ file và xây dựng từ điển cho cả tiếng Anh và tiếng Việt.

In [28]:
def load_data(file_path):
    """Đọc dữ liệu từ file, mỗi dòng là một câu"""
    with open(file_path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

# Load dữ liệu train
en_sentences = load_data('data/train.en')
vi_sentences = load_data('data/train.vi')

print(f"Số câu tiếng Anh: {len(en_sentences)}")
print(f"Số câu tiếng Việt: {len(vi_sentences)}")

# Hiển thị một vài câu mẫu
print("\n=== Ví dụ các câu song ngữ ===")
for i in range(min(5, len(en_sentences))):
    print(f"EN: {en_sentences[i]}")
    print(f"VI: {vi_sentences[i]}")
    print()

Số câu tiếng Anh: 10
Số câu tiếng Việt: 10

=== Ví dụ các câu song ngữ ===
EN: i am a student
VI: tôi là sinh viên

EN: i am a teacher
VI: tôi là giáo viên

EN: he likes football
VI: anh ấy thích bóng đá

EN: she likes music
VI: cô ấy thích âm nhạc

EN: this is a book
VI: đây là một cuốn sách



## Xây dựng từ điển cho English và Vietnamese

In [29]:
# Tạo từ điển cho tiếng Anh
vocab_en = Vocab()
vocab_en.build_vocab(en_sentences)

print("=== Từ điển tiếng Anh ===")
print(f"Tổng số tokens: {len(vocab_en)}")
print(f"Số từ vựng thực tế: {len(vocab_en) - 3} (không tính special tokens)")

print(f"\n✓ Special tokens:")
print(f"  <PAD> = {vocab_en.word2id['<PAD>']}")
print(f"  <BOS> = {vocab_en.word2id['<BOS>']}")
print(f"  <EOS> = {vocab_en.word2id['<EOS>']}")

print(f"\n10 từ thường đầu tiên (bắt đầu từ ID=3):")
regular_words = {w: idx for w, idx in vocab_en.word2id.items() 
                 if w not in ['<PAD>', '<BOS>', '<EOS>']}
for word, idx in list(regular_words.items())[:10]:
    print(f"  '{word}' -> ID: {idx}")

=== Từ điển tiếng Anh ===
Tổng số tokens: 34
Số từ vựng thực tế: 31 (không tính special tokens)

✓ Special tokens:
  <PAD> = 0
  <BOS> = 1
  <EOS> = 2

10 từ thường đầu tiên (bắt đầu từ ID=3):
  'i' -> ID: 3
  'am' -> ID: 4
  'a' -> ID: 5
  'student' -> ID: 6
  'teacher' -> ID: 7
  'he' -> ID: 8
  'likes' -> ID: 9
  'football' -> ID: 10
  'she' -> ID: 11
  'music' -> ID: 12


In [30]:
# Tạo từ điển cho tiếng Việt
vocab_vi = Vocab()
vocab_vi.build_vocab(vi_sentences)

print("=== Từ điển tiếng Việt ===")
print(f"Tổng số tokens: {len(vocab_vi)}")
print(f"Số từ vựng thực tế: {len(vocab_vi) - 3} (không tính special tokens)")

print(f"\n✓ Special tokens:")
print(f"  <PAD> = {vocab_vi.word2id['<PAD>']}")
print(f"  <BOS> = {vocab_vi.word2id['<BOS>']}")
print(f"  <EOS> = {vocab_vi.word2id['<EOS>']}")

print(f"\n10 từ thường đầu tiên (bắt đầu từ ID=3):")
regular_words = {w: idx for w, idx in vocab_vi.word2id.items() 
                 if w not in ['<PAD>', '<BOS>', '<EOS>']}
for word, idx in list(regular_words.items())[:10]:
    print(f"  '{word}' -> ID: {idx}")

=== Từ điển tiếng Việt ===
Tổng số tokens: 43
Số từ vựng thực tế: 40 (không tính special tokens)

✓ Special tokens:
  <PAD> = 0
  <BOS> = 1
  <EOS> = 2

10 từ thường đầu tiên (bắt đầu từ ID=3):
  'tôi' -> ID: 3
  'là' -> ID: 4
  'sinh' -> ID: 5
  'viên' -> ID: 6
  'giáo' -> ID: 7
  'anh' -> ID: 8
  'ấy' -> ID: 9
  'thích' -> ID: 10
  'bóng' -> ID: 11
  'đá' -> ID: 12


## Test Encoding/Decoding với dữ liệu thực

Kiểm tra xem việc encoding và decoding có hoạt động đúng không:

In [31]:
# Test với câu tiếng Anh
test_idx = 0
original_en = en_sentences[test_idx]

# Encode với special tokens (mặc định theo handout)
encoded_en = vocab_en.encode(original_en, add_special_tokens=True)
decoded_en = vocab_en.decode(encoded_en, skip_special_tokens=True)

print("=== Test English (with special tokens) ===")
print(f"Original:    '{original_en}'")
print(f"Encoded:     {encoded_en}")

# Hiển thị chi tiết
print(f"Chi tiết:    [", end="")
for i, id in enumerate(encoded_en):
    word = vocab_en.id2word[id]
    if i > 0:
        print(", ", end="")
    if word in ['<BOS>', '<EOS>']:
        print(f"\033[92m{word}\033[0m", end="")  # Màu xanh cho special tokens
    else:
        print(f"{word}", end="")
print("]")

print(f"Decoded:     '{decoded_en}'")
print(f"Match:       {original_en.lower() == decoded_en}")

print("\n✓ Theo handout: [<BOS> + words + <EOS>]")
print()

=== Test English (with special tokens) ===
Original:    'i am a student'
Encoded:     [1, 3, 4, 5, 6, 2]
Chi tiết:    [<BOS>, i, am, a, student, <EOS>]
Decoded:     'i am a student'
Match:       True

✓ Theo handout: [<BOS> + words + <EOS>]



In [32]:
# Test với câu tiếng Việt
original_vi = vi_sentences[test_idx]

# Encode với special tokens (mặc định theo handout)
encoded_vi = vocab_vi.encode(original_vi, add_special_tokens=True)
decoded_vi = vocab_vi.decode(encoded_vi, skip_special_tokens=True)

print("=== Test Vietnamese (with special tokens) ===")
print(f"Original:    '{original_vi}'")
print(f"Encoded:     {encoded_vi}")

# Hiển thị chi tiết
print(f"Chi tiết:    [", end="")
for i, id in enumerate(encoded_vi):
    word = vocab_vi.id2word[id]
    if i > 0:
        print(", ", end="")
    if word in ['<BOS>', '<EOS>']:
        print(f"\033[92m{word}\033[0m", end="")  # Màu xanh cho special tokens
    else:
        print(f"{word}", end="")
print("]")

print(f"Decoded:     '{decoded_vi}'")
print(f"Match:       {original_vi.lower() == decoded_vi}")

=== Test Vietnamese (with special tokens) ===
Original:    'tôi là sinh viên'
Encoded:     [1, 3, 4, 5, 6, 2]
Chi tiết:    [<BOS>, tôi, là, sinh, viên, <EOS>]
Decoded:     'tôi là sinh viên'
Match:       True


## Encode toàn bộ dataset với Special Tokens

Chuyển tất cả các câu thành dạng số (IDs) **bao gồm `<BOS>` và `<EOS>`** để sử dụng trong các bài lab tiếp theo (Seq2Seq, Attention, Transformer):

In [33]:
# Encode tất cả câu với special tokens (mặc định theo handout)
encoded_en_sentences = [vocab_en.encode(sent, add_special_tokens=True) for sent in en_sentences]
encoded_vi_sentences = [vocab_vi.encode(sent, add_special_tokens=True) for sent in vi_sentences]

print(f"✓ Đã encode {len(encoded_en_sentences)} câu tiếng Anh")
print(f"✓ Đã encode {len(encoded_vi_sentences)} câu tiếng Việt")
print(f"\nMỗi câu đã được thêm <BOS> (ID=1) ở đầu và <EOS> (ID=2) ở cuối")

# Hiển thị một vài ví dụ
print("\n" + "="*70)
print("=== Ví dụ câu đã encode (với special tokens) ===")
print("="*70)
for i in range(3):
    print(f"\n📌 Cặp {i+1}:")
    print(f"  EN text: {en_sentences[i]}")
    print(f"  EN IDs:  {encoded_en_sentences[i]}")
    print(f"          [{vocab_en.decode(encoded_en_sentences[i], skip_special_tokens=False)}]")
    print(f"  VI text: {vi_sentences[i]}")
    print(f"  VI IDs:  {encoded_vi_sentences[i]}")
    print(f"          [{vocab_vi.decode(encoded_vi_sentences[i], skip_special_tokens=False)}]")

print("\n" + "="*70)

✓ Đã encode 10 câu tiếng Anh
✓ Đã encode 10 câu tiếng Việt

Mỗi câu đã được thêm <BOS> (ID=1) ở đầu và <EOS> (ID=2) ở cuối

=== Ví dụ câu đã encode (với special tokens) ===

📌 Cặp 1:
  EN text: i am a student
  EN IDs:  [1, 3, 4, 5, 6, 2]
          [<BOS> i am a student <EOS>]
  VI text: tôi là sinh viên
  VI IDs:  [1, 3, 4, 5, 6, 2]
          [<BOS> tôi là sinh viên <EOS>]

📌 Cặp 2:
  EN text: i am a teacher
  EN IDs:  [1, 3, 4, 5, 7, 2]
          [<BOS> i am a teacher <EOS>]
  VI text: tôi là giáo viên
  VI IDs:  [1, 3, 4, 7, 6, 2]
          [<BOS> tôi là giáo viên <EOS>]

📌 Cặp 3:
  EN text: he likes football
  EN IDs:  [1, 8, 9, 10, 2]
          [<BOS> he likes football <EOS>]
  VI text: anh ấy thích bóng đá
  VI IDs:  [1, 8, 9, 10, 11, 12, 2]
          [<BOS> anh ấy thích bóng đá <EOS>]



## Thống kê và phân tích

Một số thống kê về dữ liệu:

In [34]:
import numpy as np

# Độ dài câu tiếng Anh (bao gồm <BOS> và <EOS>)
en_lengths = [len(sent) for sent in encoded_en_sentences]
print("=== Thống kê câu tiếng Anh ===")
print(f"Độ dài trung bình: {np.mean(en_lengths):.2f} tokens (bao gồm <BOS>, <EOS>)")
print(f"Độ dài min: {np.min(en_lengths)} tokens")
print(f"Độ dài max: {np.max(en_lengths)} tokens")
print(f"Tổng vocab size: {len(vocab_en)} tokens")
print(f"  - Special tokens: 3 (<PAD>, <BOS>, <EOS>)")
print(f"  - Regular words: {len(vocab_en) - 3}")

print()

# Độ dài câu tiếng Việt (bao gồm <BOS> và <EOS>)
vi_lengths = [len(sent) for sent in encoded_vi_sentences]
print("=== Thống kê câu tiếng Việt ===")
print(f"Độ dài trung bình: {np.mean(vi_lengths):.2f} tokens (bao gồm <BOS>, <EOS>)")
print(f"Độ dài min: {np.min(vi_lengths)} tokens")
print(f"Độ dài max: {np.max(vi_lengths)} tokens")
print(f"Tổng vocab size: {len(vocab_vi)} tokens")
print(f"  - Special tokens: 3 (<PAD>, <BOS>, <EOS>)")
print(f"  - Regular words: {len(vocab_vi) - 3}")

print("\n" + "="*70)
print("📝 LƯU Ý: Độ dài câu đã bao gồm <BOS> và <EOS>")
print("   Ví dụ: 'I am' được encode thành [1, 3, 4, 2] = 4 tokens")
print("="*70)

=== Thống kê câu tiếng Anh ===
Độ dài trung bình: 6.20 tokens (bao gồm <BOS>, <EOS>)
Độ dài min: 5 tokens
Độ dài max: 8 tokens
Tổng vocab size: 34 tokens
  - Special tokens: 3 (<PAD>, <BOS>, <EOS>)
  - Regular words: 31

=== Thống kê câu tiếng Việt ===
Độ dài trung bình: 7.30 tokens (bao gồm <BOS>, <EOS>)
Độ dài min: 6 tokens
Độ dài max: 11 tokens
Tổng vocab size: 43 tokens
  - Special tokens: 3 (<PAD>, <BOS>, <EOS>)
  - Regular words: 40

📝 LƯU Ý: Độ dài câu đã bao gồm <BOS> và <EOS>
   Ví dụ: 'I am' được encode thành [1, 3, 4, 2] = 4 tokens


## 🎯 Tóm tắt - Lab 1 Hoàn Thành Theo Handout

**Những gì đã hoàn thành theo yêu cầu `lab1_handout.tex`:**

### ✅ **Task 1: Tokenization**
```python
def tokenize(sentence):
    return sentence.lower().split()
```
Chuyển câu thành chữ thường và tách từ bằng khoảng trắng.

---

### ✅ **Task 2: Vocabulary Construction với Special Tokens**

Class `Vocab` đã triển khai đầy đủ theo handout:

**Special Tokens (theo đúng ID trong handout):**
- `<PAD>` → ID = 0 (Padding token)
- `<BOS>` → ID = 1 (Begin of Sentence)
- `<EOS>` → ID = 2 (End of Sentence)

**Các từ thường bắt đầu từ ID = 3**

**Phương thức:**
- `add_word()`: Thêm từ vào từ điển
- `build_vocab()`: Xây dựng từ điển từ corpus
- `encode(sentence, add_special_tokens=True)`: Chuyển câu → [<BOS>, word_ids..., <EOS>]
- `decode(ids, skip_special_tokens=True)`: Chuyển IDs → câu văn bản

---

### ✅ **Task 3: Sentence Encoding**

**Ví dụ encoding theo handout:**
```
Original: "I am a student"
Tokenized: ["i", "am", "a", "student"]
Encoded: [1, 3, 4, 5, 6, 2]
         ↑              ↑
       <BOS>          <EOS>
```

**Dataset đã encode:**
- ✓ Toàn bộ câu tiếng Anh với `<BOS>` và `<EOS>`
- ✓ Toàn bộ câu tiếng Việt với `<BOS>` và `<EOS>`
- ✓ Sẵn sàng cho Lab 2 (Seq2Seq), Lab 3 (Attention), Lab 4 (Transformer)

---

## 📚 Kiến thức đã học:

### **1. Tokenization (Tách từ)**
Chuyển câu thành danh sách các từ:
```
"I am a student" → ["i", "am", "a", "student"]
```

### **2. Vocabulary (Từ điển)**
Ánh xạ giữa từ và ID duy nhất:
```
<PAD> → 0
<BOS> → 1
<EOS> → 2
i     → 3
am    → 4
a     → 5
student → 6
```

### **3. Special Tokens**
- **`<PAD>`**: Đệm các câu ngắn cho bằng nhau trong batch
- **`<BOS>`**: Báo hiệu bắt đầu câu cho decoder (quan trọng trong Seq2Seq)
- **`<EOS>`**: Báo hiệu kết thúc câu, model biết khi nào dừng generate

### **4. Encoding/Decoding**
```python
# Encoding
sentence = "I am a student"
encoded = [1, 3, 4, 5, 6, 2]  # [<BOS>, i, am, a, student, <EOS>]

# Decoding
decoded = "i am a student"  # Skip special tokens
```

---

## 🚀 Sẵn sàng cho các Lab tiếp theo:

- **Lab 2: Seq2seq Model** - Encoder-Decoder architecture
- **Lab 3: Attention Mechanism** - Attention-based NMT
- **Lab 4: Transformer** - Self-attention và Multi-head attention

---

## 📝 Lưu ý quan trọng:

1. **Special tokens là bắt buộc** cho các mô hình Seq2Seq
2. **`<BOS>`** giúp decoder biết bắt đầu từ đâu
3. **`<EOS>`** giúp model biết khi nào dừng sinh câu
4. **`<PAD>`** sẽ dùng để padding trong batch training (Lab 2)

---

## ✅ Submission:
- ✓ `lab1_preprocessing.ipynb` - Hoàn thành đầy đủ theo handout
- ⏳ `report.pdf` - Cần viết báo cáo (nếu yêu cầu)

## 📚 Bonus: Subword Tokenization Concept (BPE)

### **Giới thiệu về Byte Pair Encoding (BPE)**

**BPE (Byte Pair Encoding)** là một kỹ thuật tokenization nâng cao cho phép xử lý từ vựng vô hạn bằng cách chia từ thành các **subword units** (đơn vị con).

---

### **Tại sao cần BPE?**

**Vấn đề với Word-level Tokenization (như lab này):**

1. **Out-of-Vocabulary (OOV) Problem:**
   - Từ mới không có trong vocab → không encode được
   - Ví dụ: Training có "student", nhưng test có "students" → OOV

2. **Vocab quá lớn:**
   - Mỗi word form là 1 token riêng
   - Ví dụ: "run", "running", "runs", "runner" = 4 tokens riêng biệt
   - Không share knowledge giữa các từ liên quan

3. **Hiếm gặp words:**
   - Các từ xuất hiện ít lần khó học được representations tốt

---

### **BPE hoạt động như thế nào?**

**Ý tưởng:** Chia từ thành các **subword units** có thể tái sử dụng.

**Ví dụ cụ thể:**

**Bước 1: Khởi tạo vocab với characters**
```
Vocab = {t, h, e, l, o, w, r, d}
```

**Bước 2: Tìm cặp characters xuất hiện nhiều nhất**
```
Corpus: "the", "there", "their", "low", "lower", "lowest"
Character pairs:
  "t h" → xuất hiện 3 lần (the, there, their)
  "e r" → xuất hiện 2 lần (there, lower)
→ Merge "t" + "h" thành "th"
```

**Bước 3: Lặp lại quá trình**
```
Iteration 1: Merge "t" + "h" → "th"
Iteration 2: Merge "th" + "e" → "the"
Iteration 3: Merge "l" + "o" → "lo"
Iteration 4: Merge "lo" + "w" → "low"
...
```

**Kết quả:**
```
Vocab = {
  characters: t, h, e, l, o, w, r, d, ...
  subwords:   th, the, lo, low, er, ...
  words:      the, low, lower, ...
}
```

---

### **Ví dụ Encoding với BPE**

**Từ mới:** "unbelievable" (chưa từng thấy trong training)

**Word-level tokenization:** `<UNK>` (không biết từ này)

**BPE tokenization:**
```
"unbelievable" → ["un", "believ", "able"]
```

✓ Mỗi subword có thể đã học trong training từ các từ khác:
- "un" từ "unable", "unhappy"
- "believ" từ "believe", "believing"  
- "able" từ "able", "capable"

→ Model vẫn có thể hiểu nghĩa của từ mới!

---

### **So sánh Word-level vs BPE**

| Aspect | Word-level (Lab này) | BPE Tokenization |
|--------|---------------------|------------------|
| **Vocab size** | Lớn (mỗi word = 1 token) | Nhỏ hơn (share subwords) |
| **OOV words** | ❌ Không xử lý được | ✅ Chia thành subwords |
| **Morphology** | ❌ "run"/"running" riêng biệt | ✅ Share "run" subword |
| **Rare words** | ❌ Khó học | ✅ Dùng subwords đã biết |
| **Complexity** | ✅ Đơn giản | ⚠️ Phức tạp hơn |

---

### **Ví dụ thực tế**

**Tokenization type so sánh:**

```python
# Word-level (Lab 1)
"I'm a student" → ["i'm", "a", "student"]
Vocab: {i'm, a, student, ...}

# BPE
"I'm a student" → ["I", "'m", "a", "stud", "ent"]
Vocab: {I, 'm, a, stud, ent, ...}
```

**Từ mới:** "I'm a teacher"

```python
# Word-level
"teacher" → <UNK> (nếu chưa thấy "teacher" trong training)

# BPE
"teacher" → ["teach", "er"]
# "teach" và "er" có thể đã học từ "teaching", "learner", etc.
```

---

### **BPE trong các mô hình hiện đại**

**BPE được sử dụng rộng rãi:**

1. **GPT series (OpenAI):**
   - GPT-2, GPT-3, GPT-4, ChatGPT
   - Vocab size: ~50K BPE tokens

2. **BERT (Google):**
   - WordPiece (tương tự BPE)
   - Vocab size: 30K tokens

3. **Machine Translation:**
   - Google Translate
   - Facebook FAIR (fairseq framework)

4. **SentencePiece:**
   - Unigram LM + BPE
   - Sử dụng bởi T5, mT5, XLM-R

---

### **Thuật toán BPE (Pseudo-code)**

```python
def learn_bpe(corpus, num_merges):
    # 1. Initialize vocab với characters
    vocab = set(all_characters_in_corpus)
    
    for i in range(num_merges):
        # 2. Đếm tần suất tất cả các pairs
        pair_counts = count_all_pairs(corpus)
        
        # 3. Tìm pair xuất hiện nhiều nhất
        best_pair = max(pair_counts, key=pair_counts.get)
        
        # 4. Merge pair đó thành 1 token mới
        vocab.add(merge(best_pair))
        
        # 5. Update corpus
        corpus = replace_pair_in_corpus(corpus, best_pair)
    
    return vocab

def encode_with_bpe(word, bpe_vocab):
    # Chia word thành sequence of subwords có trong vocab
    subwords = []
    while word:
        # Tìm longest matching subword
        for length in range(len(word), 0, -1):
            prefix = word[:length]
            if prefix in bpe_vocab:
                subwords.append(prefix)
                word = word[length:]
                break
    return subwords
```

---

### **Ưu điểm của BPE**

✅ **Open vocabulary:** Không có OOV problem
✅ **Smaller vocab size:** Hiệu quả hơn với bộ nhớ
✅ **Better for morphologically rich languages:** Tiếng Đức, Tiếng Thổ Nhĩ Kỳ
✅ **Handles rare words:** Sử dụng subwords đã biết
✅ **State-of-the-art:** Dùng trong tất cả LLMs hiện đại

---

### **Nhược điểm của BPE**

❌ **Phức tạp hơn:** Cần training BPE model trước
❌ **Tokenization không trực quan:** "hello" có thể thành ["he", "llo"]
❌ **Sequence dài hơn:** Nhiều subwords hơn số words
❌ **Language-specific:** Cần BPE riêng cho mỗi ngôn ngữ

---

### **Khi nào dùng Word-level vs BPE?**

**Word-level (như Lab 1):**
- ✅ Dataset nhỏ, vocab giới hạn
- ✅ Học tập, prototyping
- ✅ Ngôn ngữ có boundaries rõ ràng

**BPE:**
- ✅ Production systems
- ✅ Large-scale models
- ✅ Multilingual models
- ✅ Morphologically rich languages

---

### **Tài liệu tham khảo**

📄 **Papers:**
- Sennrich et al. (2016) - "Neural Machine Translation of Rare Words with Subword Units"
- Kudo & Richardson (2018) - "SentencePiece: A simple and language independent approach"

🔧 **Libraries:**
- `sentencepiece` - Google's implementation
- `tokenizers` - HuggingFace (fast BPE implementation)
- `subword-nmt` - Original BPE implementation

---

### **📝 Kết luận**

Trong **Lab 1 này**, chúng ta sử dụng **word-level tokenization** vì:
1. ✅ Đơn giản, dễ hiểu
2. ✅ Đủ cho dataset nhỏ (10 câu)
3. ✅ Focus vào concept cơ bản của NLP preprocessing

Tuy nhiên, **BPE là chuẩn mực** cho production NMT systems và được sử dụng trong:
- 🚀 GPT-4, ChatGPT
- 🚀 BERT, RoBERTa
- 🚀 Google Translate
- 🚀 Tất cả modern LLMs

**Trong các dự án thực tế, bạn nên sử dụng BPE!**